# Measuring Cycle Shapes with Planar Loops

A cycle representative of a 2D image runs along pixel edges. Its staircase is
exact for counting pixels, but it distorts every measurement that involves its
length or its turning: a digital disk has the perimeter of its bounding box,
so its circularity tends to $\pi^2/16 \approx 0.62$ instead of 1, and every
step of the staircase adds a right-angle turn to its curvature.

This tutorial turns representatives into *planar loops* first and measures
those:

- `cycle_loops(chain, forest.geometry, curve=..., scale=...)` from
  `persforest.geometry.loops` builds `PlanarLoops`, oriented closed polylines
  in physical coordinates:
  - `curve="raw"` follows the representative: the staircase of a cubical
    cycle, the polygon of a simplicial one.
  - `curve="midpoint"` (cubical only) joins the midpoints of consecutive grid
    edges, cutting every staircase corner.
  - `curve="smoothed"` applies Gaussian smoothing with arc-length standard
    deviation `scale` (physical units, required) to the midpoint loop, or to
    the polygon of a point-cloud cycle, and keeps the enclosed area of the
    raw cycle.
- The functionals of `persforest.measurements.loops` (`loops_length`,
  `loops_area`, `loops_circularity`, `loops_excess_curvature`, convex-hull
  deficits, ...) measure `PlanarLoops`.
- `LoopMeasurement(functional, curve=..., scale=...)` packages both steps as a
  `(chain, geometry)` measurement for profiles and landscapes.
- `plot_planar_loops` from `persforest.plotting.loops` draws loops on their
  own or on top of a forest plot.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from persforest import PersistenceForest
from persforest.geometry.loops import cycle_loops
from persforest.geometry.planar import polygon_area
from persforest.measurements.loops import (
    LoopMeasurement, loops_area, loops_circularity, loops_excess_curvature, loops_length,
)
from persforest.plotting.loops import plot_planar_loops

SPACING = 0.05  # physical size of a pixel


def peaks_image(shape=(150, 100)):
    # Three peaks on a flat background, built with (axis 0, axis 1) coordinates.
    # Every level set of a peak is the same shape, scaled: a disk, a 2:1
    # ellipse rotated by 30 degrees and a square rotated by 20 degrees.
    x, y = (np.mgrid[0:shape[0], 0:shape[1]] + 0.5) * SPACING

    def rotated(center, degrees):
        c, s = math.cos(math.radians(degrees)), math.sin(math.radians(degrees))
        return c * (x - center[0]) + s * (y - center[1]), -s * (x - center[0]) + c * (y - center[1])

    u, v = rotated((1.3, 2.5), 0)
    disk = 1 - np.hypot(u, v) / 1.1
    u, v = rotated((3.75, 1.45), 30)
    ellipse = 1 - np.hypot(u / 1.1, v / 0.55)
    u, v = rotated((5.9, 3.1), 20)
    square = 1 - np.maximum(abs(u), abs(v)) / 0.9
    return np.maximum.reduce([disk, ellipse, square, np.zeros(shape)])


def ellipse_circularity(a, b):
    # Ramanujan's perimeter approximation; accurate to 1e-5 for a 2:1 ellipse.
    perimeter = math.pi * (3 * (a + b) - math.sqrt((3 * a + b) * (a + 3 * b)))
    return 4 * math.pi * math.pi * a * b / perimeter**2


PEAKS = {  # center and true circularity of every level set
    "disk": ((1.3, 2.5), 1.0),
    "ellipse": ((3.75, 1.45), ellipse_circularity(2, 1)),
    "square": ((5.9, 3.1), math.pi / 4),
}

## A forest with three known shapes

In the sublevel filtration the background enters at 0, closing a loop around
every peak; the loop shrinks onto the peak as the filtration value grows. All
level sets of a peak have the same shape, so a faithful circularity stays
constant along each bar.

In [ ]:
image = peaks_image()
forest = PersistenceForest.from_cubical_grid(image, spacing=SPACING, storage="cycles")


def peak_of(bar, value=0.3):
    center = cycle_loops(forest.cycle_for_bar_at(bar, value), forest.geometry).points.mean(axis=0)
    return min(PEAKS, key=lambda name: np.hypot(*(np.asarray(PEAKS[name][0]) - center)))


bars = {peak_of(bar): bar for bar in forest.longest_bars(3)}
for name, bar in bars.items():
    print(f"{name:8s} bar [{bar.birth:.3f}, {bar.death:.3f})")

T = 0.3
forest.plot_at_filtration(T, figsize=(7, 5), show=False, title=f"representatives at {T}")
plt.show()

## From a representative to loops

`cycle_loops` returns `PlanarLoops`; `plot_planar_loops` draws them on top of
`plot_at_filtration` with `show_cycles=False`. Here are the three curves of
the ellipse's representative; `scale=2 * SPACING` smooths over two pixels.
The domain outline is switched off because the panels are zoomed in.

In [ ]:
ZOOM_STYLE = {"domain_edge_width": 0, "image_colorbar": False}


def zoom_to(ax, loops, margin=0.1):
    lower, upper = loops.points.min(axis=0) - margin, loops.points.max(axis=0) + margin
    ax.set_xlim(lower[0], upper[0])
    ax.set_ylim(lower[1], upper[1])


rep = forest.cycle_for_bar_at(bars["ellipse"], T)
curves = [("raw", None), ("midpoint", None), ("smoothed", 2 * SPACING)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (curve, scale) in zip(axes, curves):
    loops = cycle_loops(rep, forest.geometry, curve, scale)
    title = f"curve={curve!r}" + (f", scale={scale:g}" if scale else "")
    forest.plot_at_filtration(T, ax=ax, show=False, show_cycles=False, style_2d=ZOOM_STYLE, title=title)
    plot_planar_loops(loops, ax=ax, show=False, color="crimson", show_vertices=curve != "smoothed")
    zoom_to(ax, loops)
fig.tight_layout()
plt.show()

loops = cycle_loops(rep, forest.geometry, "smoothed", 2 * SPACING)
print(loops.kind, dict(loops.params), f"{len(loops)} loop with {len(loops.loops[0])} samples")

## What the curves measure

Circularity $4\pi A / L^2$ and excess curvature (total absolute turning minus
$2\pi$, zero for every convex curve) of each representative at value 0.3,
next to the true circularity of its level sets.

- The **staircase** puts every shape near 0.5 and cannot tell the square
  from the ellipse; its curvature counts the turns of the grid.
- The **midpoint** loop recovers most of the length (a few percent too long)
  but still turns at every step.
- The **smoothed** loop at two pixels is within about 0.01 of the true
  circularity for the disk and the ellipse, with essentially no excess
  curvature. The square comes out rounder than it is: smoothing rounds its
  corners over the scale.

Excess curvature of staircase and midpoint loops emits a `UserWarning` for
exactly this reason; the table silences it.

In [ ]:
variants = {"raw": ("raw", None), "midpoint": ("midpoint", None),
            "1 px": ("smoothed", SPACING), "2 px": ("smoothed", 2 * SPACING)}
print(f"{'':8s} {'true':>6s} |" + "".join(f"{name:>9s}" for name in variants) + "  | excess curvature (rad)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    for name, bar in bars.items():
        rep = forest.cycle_for_bar_at(bar, T)
        loops = [cycle_loops(rep, forest.geometry, curve, scale) for curve, scale in variants.values()]
        print(f"{name:8s} {PEAKS[name][1]:6.3f} |"
              + "".join(f"{loops_circularity(l):9.3f}" for l in loops) + "  |"
              + "".join(f"{loops_excess_curvature(l):8.2f}" for l in loops))

rep = forest.cycle_for_bar_at(bars["disk"], T)
print(f"\ndisk at {T}: {len(rep.signed_simplices)} grid edges")
for curve, scale in variants.values():
    loops = cycle_loops(rep, forest.geometry, curve, scale)
    print(f"  {curve:8s} scale={scale!s:5s} length {loops_length(loops):.4f}  area {loops_area(loops):.4f}")

The area column shows the bookkeeping: the staircase encloses the pixel area
exactly, the midpoint loop loses half a pixel ($\tfrac12$ `spacing[0] * spacing[1]`)
per loop, and the smoothed loop is rescaled to the pixel area again. Only the
length changes, and with it the circularity.

## Choosing the scale

`scale` is a modelling choice and therefore required. Small scales leave the
pixel steps in the curve (the curvature column), large scales round corners
and, once the scale approaches the size of a feature, remove it. Two pixels
removes the steps; anything larger should be justified by the size of the
shapes of interest.

In [ ]:
rep = forest.cycle_for_bar_at(bars["square"], T)
scales = [0.5 * SPACING, SPACING, 2 * SPACING, 4 * SPACING, 8 * SPACING]
fig, ax = plt.subplots(figsize=(6, 5))
forest.plot_at_filtration(T, ax=ax, show=False, show_cycles=False, style_2d=ZOOM_STYLE,
                          title="square: smoothed at increasing scale")
colors = plt.cm.viridis(np.linspace(0, 0.9, len(scales)))
for scale, color in zip(scales, colors):
    loops = cycle_loops(rep, forest.geometry, "smoothed", scale)
    plot_planar_loops(loops, ax=ax, show=False, color=color, linewidth=1.5, label=f"{scale / SPACING:g} px")
    print(f"scale {scale / SPACING:3g} px: circularity {loops_circularity(loops):.3f}, "
          f"excess curvature {loops_excess_curvature(loops):5.2f}")
zoom_to(ax, cycle_loops(rep, forest.geometry))
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), title="scale")
plt.show()

## Small cycles

On a grid a cycle cannot be shorter than the four edges of one pixel. The
midpoint loop of a pixel is a diamond with perimeter $2\sqrt{2}$, and smoothing
turns it into the circle of the same area, perimeter $2\sqrt{\pi}$. A 1×3 bar
of pixels becomes an ellipse-like loop of area 3.

In [ ]:
small = np.zeros((12, 8))
small[3, 3] = 1      # one pixel
small[6:9, 3] = 1    # a 1x3 bar along axis 0
small_forest = PersistenceForest.from_cubical_grid(small, storage="cycles")

fig, ax = plt.subplots(figsize=(7, 4.5))
small_forest.plot_at_filtration(0.5, ax=ax, show=False, show_cycles=False, title="pixel and 1x3 bar")
small_bars = sorted((bar for bar in small_forest.barcode if bar.birth <= 0.5 < bar.death), key=lambda bar: bar.birth)
for index, bar in enumerate(small_bars):
    rep = small_forest.cycle_for_bar_at(bar, 0.5)
    for (curve, scale), color in zip([("raw", None), ("midpoint", None), ("smoothed", 1.0)],
                                     ["tab:blue", "tab:orange", "crimson"]):
        loops = cycle_loops(rep, small_forest.geometry, curve, scale)
        plot_planar_loops(loops, ax=ax, show=False, color=color, label=curve if index == 0 else None)
        print(f"{len(rep.signed_simplices)} edges, {curve:8s}: length {loops_length(loops):.4f}, "
              f"area {loops_area(loops):.3f}, circularity {loops_circularity(loops):.3f}")
ax.legend(loc="upper right")
plt.show()

## Holes and orientation

A representative bounds one connected component of the complement of the
complex. When that component has holes (islands of the complex inside it),
its boundary has several loops: the outer one runs counterclockwise
(positive signed area) and each hole boundary clockwise (negative signed
area). `loops_area` sums the signed areas, which is the outer area minus the
hole areas. Here a bright ring encloses a dark island; `show_orientation=True`
marks the direction of each loop.

In [ ]:
x, y = (np.mgrid[0:60, 0:60] + 0.5) * SPACING - 1.5
radius = np.hypot(x, 1.4 * y)
ring = ((radius < 1.2) & (radius > 0.45)).astype(float)
ring_forest = PersistenceForest.from_cubical_grid(ring, spacing=SPACING, storage="cycles")
[ring_bar] = [bar for bar in ring_forest.barcode if bar.death - bar.birth > 0.5]
rep = ring_forest.cycle_for_bar_at(ring_bar, 0.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (curve, scale) in zip(axes, [("raw", None), ("smoothed", 2 * SPACING)]):
    loops = cycle_loops(rep, ring_forest.geometry, curve, scale)
    ring_forest.plot_at_filtration(0.5, ax=ax, show=False, show_cycles=False, title=f"curve={curve!r}",
                                   style_2d={"image_colorbar": False})
    plot_planar_loops(loops, ax=ax, show=False, color="crimson", show_orientation=False, linewidth=2)
    signed = [polygon_area(loop, signed=True) for loop in loops]
    print(f"{curve:8s}: signed loop areas {np.round(signed, 4)} -> loops_area {loops_area(loops):.4f}")
print(f"pixel count x pixel area: {ring.sum() * SPACING**2:.4f}")
plt.show()

## Measurement profiles and landscapes

`LoopMeasurement` turns a loop functional into the `(chain, geometry)`
measurement that profiles and landscapes expect. Its `name` and `params`
describe it; pass them on explicitly. Profiles use the bar colors of the
first figure (disk orange, ellipse blue, square green); dashed lines mark the true
circularities: the raw profiles saw-tooth with every added pixel and do not
separate the square from the ellipse, while the smoothed ones stay near their
true values. The square drifts toward 1 at the end of its bar, where it is
only a few pixels wide and the fixed two-pixel scale rounds it off.

In [ ]:
measurements = [LoopMeasurement(loops_circularity),
                LoopMeasurement(loops_circularity, curve="smoothed", scale=2 * SPACING)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, measurement in zip(axes, measurements):
    print(repr(measurement))
    profiles = forest.compute_measurement_profiles(
        measurement, measurement.name, min_bar_length=0.5,
        measurement_name=measurement.name, measurement_params=measurement.params)
    forest.plot_measurement_profiles(profiles, ax=ax, min_bar_length=0.5, title=measurement.name,
                                     ylabel="circularity")
    for _, circularity in PEAKS.values():
        ax.axhline(circularity, color="0.5", linewidth=0.8, linestyle="--")
    ax.set_ylim(0, 1.05)
plt.show()

Landscapes take the same measurement object.

In [ ]:
smooth_circularity = measurements[1]
forest.compute_measurement_landscapes(smooth_circularity, "smoothed circularity", max_k=3,
                                      min_bar_length=0.5, measurement_params=smooth_circularity.params)
forest.plot_measurement_landscapes("smoothed circularity")
plt.show()

## Point clouds

`curve="smoothed"` also applies to cycles of point clouds; it smooths the
polygon of the representative (there is no midpoint curve). The scale is in
the units of the points. Below, an alpha complex of a noisy 2:1 ellipse shortly
after its loop closes: the raw polygon zig-zags between sample points, which
the smoothed loop removes while keeping the enclosed area. The representative
runs along the inner side of the noisy band, so it encloses less than the
ellipse through the sample centers.

In [ ]:
rng = np.random.default_rng(0)
angles = rng.uniform(0, 2 * np.pi, 250)
points = np.column_stack([2 * np.cos(angles), np.sin(angles)]) + rng.normal(scale=0.06, size=(250, 2))
cloud_forest = PersistenceForest.from_points(points, storage="cycles")
bar = cloud_forest.max_bar()
value = bar.birth + 0.05 * (bar.death - bar.birth)
rep = cloud_forest.cycle_for_bar_at(bar, value)

fig, ax = plt.subplots(figsize=(8, 4.5))
cloud_forest.plot_at_filtration(value, ax=ax, show=False, show_cycles=False, title="alpha complex of a noisy ellipse")
for (curve, scale), color in zip([("raw", None), ("smoothed", 0.15)], ["tab:blue", "crimson"]):
    loops = cycle_loops(rep, cloud_forest.geometry, curve, scale)
    plot_planar_loops(loops, ax=ax, show=False, color=color, linewidth=1.8, label=curve)
    print(f"{curve:8s}: circularity {loops_circularity(loops):.3f}, area {loops_area(loops):.3f}")
print(f"ellipse 2:1: circularity {ellipse_circularity(2, 1):.3f}")
ax.legend(loc="upper right")
plt.show()

## Summary

| step | function |
| --- | --- |
| representative → loops | `cycle_loops(chain, forest.geometry, curve, scale)` |
| measure loops | `loops_length`, `loops_area`, `loops_circularity` (and complements), `loops_non_circularity`, `loops_convex_hull_area_deficit`, `loops_convex_hull_perimeter_deficit`, `loops_max_convexity_defect_depth`, `loops_excess_curvature` |
| profiles and landscapes | `LoopMeasurement(functional, curve=..., scale=..., **kwargs)` |
| plot | `plot_planar_loops(loops, ax=..., image_orientation=...)` |

Midpoint and smoothed loops are built from `chain.unsigned()`: they describe
the region the cycle bounds. Tendrils, which appear on grids with
`filtration_type="vertices"`, are measured on the chain itself with
`cubical_chain_avg_tendril_length`, `cubical_chain_num_of_branching_points`
and the other `cubical_chain_*` functionals of `persforest.measurements.cubical`.
When plotting with `image_orientation=True`, pass the same flag to
`plot_planar_loops`.